# 04 · Export model bundles for the phone

Turns each trained checkpoint into a **model bundle** (`exports/<name>.unetmodel`): the Core ML
model, a model card and a self-test, following `docs/model_contract.md`. The app imports
bundles into its model library, and you switch between them on the Inference tab.

This notebook is a thin wrapper around the command-line tool:

    python -m unetultra.export --checkpoint models/model.pt --name unet-multi-512 --out exports/

Run it on the Mac: checking Core ML against PyTorch needs macOS. Run `03_evaluate` first so the
test metrics are copied into each model card.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "python"))  # use the package without installing it
DATA = Path("../data")      # session zips exported from the phone
MODELS = Path("../models")  # checkpoints, calibration.json, UNetUltra.mlpackage
MODELS.mkdir(exist_ok=True)

In [ ]:
# checkpoint name -> (bundle name, name shown in the app)
BUNDLES = {
    "model": ("unet-multi-512", "U-Net + focus"),
    "model_image_only": ("unet-image-only-512", "U-Net, image only"),
    "model_embed": ("unet-embed-dinov2s-512", "U-Net + DINOv2-S"),
}
PRECISION = "fp16"   # "fp32" is about twice the size and matches PyTorch more closely
EXPORTS = Path("../exports")

In [ ]:
import json, shutil
from unetultra.calibration import LensCalibration
from unetultra.dataset import SessionDataset
from unetultra.export import export_bundle
from unetultra.models import uses_meta
from unetultra.session import find_sessions
from unetultra.train import load_model

calibration = LensCalibration.load(MODELS / "calibration.json")
zips = find_sessions(DATA, "test") or find_sessions(DATA)
cards = {}
for checkpoint_name, (bundle_name, display_name) in BUNDLES.items():
    checkpoint_path = MODELS / f"{checkpoint_name}.pt"
    if not checkpoint_path.exists():
        print("skipping", checkpoint_name, "(not trained)")
        continue
    architecture = load_model(checkpoint_path)[1]["architecture"]
    samples = SessionDataset(zips, calibration, use_meta=uses_meta(architecture))
    metrics_file = MODELS / f"{checkpoint_name}_test_metrics.json"
    test_metrics = json.loads(metrics_file.read_text()) if metrics_file.exists() else None
    bundle, cards[bundle_name], comparison = export_bundle(
        checkpoint_path, bundle_name, EXPORTS, samples, PRECISION, display_name, test_metrics)
    if comparison:
        worst = max(row["depth_max_abs_diff_m"] for row in comparison)
        changed = max(row["mask_pixels_changed"] for row in comparison)
        tolerance = 0.01 if PRECISION == "fp16" else 0.001
        print("  PASS" if worst < tolerance else "  CHECK", f"worst depth difference {worst * 1000:.2f} mm, "
              f"at most {changed} mask pixels changed\n")
shutil.copy(MODELS / "calibration.json", EXPORTS / "calibration.json")

## What was exported

In [ ]:
for name, card in cards.items():
    print(f"{card['display_name']:22} {card['architecture']:12} {card['size_mb']:6.1f} MB  {card['precision']}  "
          f"meta {card['meta_features']}  test {card['test_metrics']}")
print("\nFiles in", EXPORTS.resolve())
for path in sorted(EXPORTS.iterdir()):
    print("  ", path.name)

## Send to the phone

AirDrop the `.unetmodel` files and `calibration.json` from `exports/` to the iPhone (or put them
in iCloud Drive). Open each `.unetmodel` with UNet-ultra, or import it from the app's Models
screen; import `calibration.json` once from the Inference tab. Each import runs the bundle's
self-test on the phone and shows pass or fail. Nothing in `exports/` is committed to git.